# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, rdMolDescriptors, MACCSkeys
from itertools import combinations
import time
# from rdkit import Chem, DataStructs
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from rdkit.Chem import Descriptors
desc_list = [f for name, f in Descriptors.descList]

In [ ]:
# Input file with chemical names
file_path = '../validation_data.xlsx'
# Output path where we save the new excel files with likeness scores ready for ML training
output_path = './full_descriptors/validation/'
# import data frame for pre-processing
df = pd.read_excel(file_path)
df.head()

# Data pre-processing

In [ ]:
# Define mapping for 'solv_ratio' and apply if present
type_mapping = {'100/0': 1, '75/25': 0.75, '80/20': 0.8, '70/30': 0.7, '60/40': 0.6, '50/50': 0.5}
if 'solv_ratio' in df.columns:
    df['solv_ratio'] = df['solv_ratio'].replace(type_mapping)

# Define mapping for 'solv_ratio' and apply if present
type_mapping = {"100/0": 1, '80/20': 0.8, '60/40': 0.6, '50/50': 0.5, '40/60': 0.4, '20/80': 0.2}
if 'comp1_frac' in df.columns:
    df['comp1_frac'] = df['comp1_frac'].replace(type_mapping)

# Initialize the SMILES maps
# Monomer unit
smiles_monomer_map = {
    "PS": "C=CC1=CC=CC=C1",               # Polystyrene
    "PEO": "C=CO",                 # Poly(ethylene oxide)
    "P2VP": "C=CC1=CC=CC=N1",    # Poly(2-vinylpyridine) 
    "P4VP": "C=CC1=CC=NC=C1",    # Poly(4-vinylpyridine)
    "PMMA": "CC(=C)C(=O)OC",    # Poly(methyl methacrylate)
    "PIB": "CC(=C)C",    # Poly(isobutylene)
    "PVA": "CC(=O)OC=C ",    # Poly(vinyl acetate) 
    "PCL": "OCCCCCC(=O)",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "CC(=C)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "CC(=C)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "C=CC#N",    # Poly(acrylonitrile)
    "PtBuA": "CC(C)(C)OC(=O)C=C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(=C)C",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}

# Repeat unit with polymerization points
smiles_poly_map = {
    "PS": "*CC(C1=CC=CC=C1)*",               # Polystyrene (repeat unit) example
    "PEO": "*CCO*",                 # Poly(ethylene oxide)
    "P2VP": "*CC(c1ccccn1)*",    # Poly(2-vinylpyridine) (monomer unit approx)
    "P4VP": "*CC(c1ccncc1)*",    # Poly(4-vinylpyridine)
    "PMMA": "*C(C)(C(=O)OC)*",    # Poly(methyl methacrylate)
    "PIB": "*C(C)(C)C*",    # Poly(isobutylene)
    "PVA": "*C(OC(=O)C)C*",    # Poly(vinyl acetate) 
    "PCL": "*OCCCCCC(=O)*",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "*C(C)(C*)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "*CC(C)(*)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "*CC(C#N)*",    # Poly(acrylonitrile)
    "PtBuA": "*CC(*)C(=O)OC(C)(C)C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(C)(*)C*",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}

# Repeat unit
smiles_repeat_map = {
    "PS": "CC(C1=CC=CC=C1)",               # Polystyrene (repeat unit) example
    "PEO": "CCO",                 # Poly(ethylene oxide)
    "P2VP": "CC(c1ccccn1)",    # Poly(2-vinylpyridine) (monomer unit approx)
    "P4VP": "CC(c1ccncc1)",    # Poly(4-vinylpyridine)
    "PMMA": "C(C)(C(=O)OC)",    # Poly(methyl methacrylate)
    "PIB": "C(C)(C)C",    # Poly(isobutylene)
    "PVA": "C(OC(=O)C)C",    # Poly(vinyl acetate) 
    "PCL": "OCCCCCC(=O)",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "C(C)(C)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "CC(C)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "CC(C#N)",    # Poly(acrylonitrile)
    "PtBuA": "CCC(=O)OC(C)(C)C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(C)C",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}


# Columnds with chemical information that needs to be converted to SMILES
chem_cols = [
    "comp1_blockA_type", "comp1_blockB_type",
    "comp2_blockA_type", "comp2_blockB_type",
    "add_type", "solv_1", "solv_2"
]

In [ ]:
# Create a master smiles map containing the 3 maps
smiles_maps = [
    ("poly",   smiles_poly_map),
    ("repeat", smiles_repeat_map),
    ("monomer", smiles_monomer_map)
]

df_results = {}  # store each DataFrame by a key

# For each SMILES map, create a new df that replaces chemical names with SMILES structures
for name, smap in smiles_maps:
    # Work on a copy so originals aren’t clobbered
    temp_df = df.copy()
    for col in chem_cols:
        if col in temp_df.columns:
            temp_df[col] = temp_df[col].map(smap).fillna(temp_df[col])
    df_results[name] = temp_df # df_results contains multiple dfs, one for each SMILES map

In [ ]:
# Mordred calculator
mordred_calc = Calculator(descriptors, ignore_3D=True)

# Converts the SMILES structures to a variety of fingerprinting/descriptor methods
def compute_descriptors(smiles, method="morgan", **kwargs):
    """
    Returns a full fingerprint/descriptor vector for a single SMILES string.
    """
    if pd.isna(smiles) or smiles == "":
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    method = method.lower()

    if method == "morgan":
        radius = kwargs.get("radius", 2)
        nBits = kwargs.get("nBits", 2048)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
        arr = np.zeros((nBits,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.tolist()

    elif method == "rdkit_path":
        nBits = kwargs.get("nBits", 2048)
        minPath = kwargs.get("minPath", 1)
        maxPath = kwargs.get("maxPath", 7)
        fp = Chem.RDKFingerprint(mol, fpSize=nBits, minPath=minPath, maxPath=maxPath)
        arr = np.zeros((nBits,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.tolist()

    elif method == "maccs":
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((len(fp),), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.tolist()

    elif method == "rdkit_physchem":
        from rdkit.Chem import Descriptors
        desc_list = [f for name, f in Descriptors.descList]
        return [f(mol) for f in desc_list]

    elif method == "mordred":
        try:
            d = mordred_calc(mol).fill_missing(0)
            return list(d.values())
        except Exception:
            return None

    else:
        raise ValueError(f"Unknown method: {method}")

# Compute all descriptors/fingerprints for each SMILES in a row
def calculate_descriptors_row(row, method):
    results = {}

    for col in smiles_cols: 
        smiles = row.get(col)
        desc = compute_descriptors(smiles, method=method)

        if desc is None:
            # Keep a flag for missing or invalid SMILES
            results[f"{col}_{method}_exists"] = 0
            continue
        else:
            results[f"{col}_{method}_exists"] = 1

        # Store each descriptor/fingerprint value under its own column name
        if isinstance(desc, (list, np.ndarray)):
            for i, val in enumerate(desc):
                results[f"{col}_{method}_desc_{i}"] = val
        else:
            # Shouldn’t happen, but in case a scalar slips through
            results[f"{col}_{method}_desc_0"] = desc

    return pd.Series(results)

# The resulting df is messy and difficult to parse, this function reorganizes the df, specific to this df (Not generalizable)
def column_reoganizer(df):
    # Rearrange the columns for easier analysis down the line
    cols = list(df.columns)
    # First column is related to the sample ids (Keep for troubleshooting)
    sample_ids = cols[0:1]
    # Out generated likeness descriptors fall at the end of the sheet
    likeness_descriptors = cols[17:]
    # Processing conditions are jumbled up due to original formatting
    mw_descriptors = [cols[i] for i in [2, 4, 6, 8] if i < len(cols)]
    processing_desciptors_1 = cols[9:min(11,len(cols))]
    processing_desciptors_2 = cols[12:min(13,len(cols))]
    processing_descriptors = list(processing_desciptors_1) + list(processing_desciptors_2) + list(mw_descriptors)
    # Outputs are at the end of the original df
    outputs = cols[13:min(15,len(cols))]
    # Combine all of these into one list of column names
    new_cols = list(sample_ids) + list(likeness_descriptors) + processing_descriptors + list(outputs)
    # Create a new organized df
    df_organized = df.loc[:, new_cols]  # loc ensures we’re selecting by labels
    return df_organized

In [ ]:
%%time

# Similarity methods we are using
similarity_methods = ["morgan", "rdkit_path", "maccs", "rdkit_physchem", "mordred"]
# Generate all column pairs that may contain SMILES structures
smiles_cols = chem_cols
df_likeness = {}
for name, smap in smiles_maps:
    for method in similarity_methods:
        # Continuous descriptor methods cannot handle the wildcard present in the 'poly' SMILES
        if "poly" in name.lower() and method in ["rdkit_physchem", "mordred"]:
            continue

        df_likeness_key = f"{name}_{method}"
        
        print(f'\nGenerating descriptors for: {df_likeness_key}')
        start_time = time.time()
        # Compute likeness for this method
        descriptors_df = df_results[name].apply(calculate_descriptors_row, axis=1, method=method)

        # Combine original with likeness scores
        temp_df = pd.concat([df_results[name], descriptors_df], axis=1)
        
        # Reorganize
        temp_df = column_reoganizer(temp_df)

        # Identify all columns except the last two (Selects all of our input columns)
        cols_to_fill = temp_df.columns[:-2]      

        # Fill NaN values in these columns with 0 (These are cases where the likeness pair does not exist)
        temp_df[cols_to_fill] =  temp_df[cols_to_fill].fillna(0)

        df_likeness[df_likeness_key] = temp_df

        # Save to distinct Excel
        df_likeness[df_likeness_key].to_excel(f"{output_path}{name}_{method}.xlsx", index=False)
        end_time = time.time()

        elapsed = (end_time - start_time)/60
        print(f"Time for {df_likeness_key}: {elapsed:.2f} minutes")